# Risk Parity
Risk parity's whole premise is: **throw away $\mu$ entirely.** Don't estimate expected returns at all; allocate so that every asset contributes *equal risk* to the portfolio. It's the answer to "what if I only trust $\Sigma$, not $\mu$?"

In [1]:
import cvxpy as cp
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from util import util_yahoo_finance as uyf

Portfolio variance is $\sigma_p^2 = w^T\Sigma w$. Differentiate with respect to the weights:

$$
\frac{\partial \sigma_p^2}{\partial w} = 2\Sigma w
$$

That gradient $\Sigma w$ is *exactly* the gradient of the Markowitz objective, as the $\beta$-defining covariance-with-the-portfolio in CAPM, and as the FOC term in every optimization. The $i$-th entry $(\Sigma w)_i = \text{Cov}(r_i, r_p)$ is asset $i$'s covariance with the portfolio. Risk parity is built entirely on this vector.

The three quantities, all from $\Sigma w$:

The **marginal risk contribution** of asset $i$ — how much portfolio volatility $\sigma_p = \sqrt{w^T\Sigma w}$ changes per unit change in $w_i$:

$$
\text{MRC}_i = \frac{\partial \sigma_p}{\partial w_i} = \frac{(\Sigma w)_i}{\sigma_p}
$$

The **total risk contribution** of asset $i$ (its share of portfolio risk), weighting the marginal by the position size:

$$
\text{RC}_i = w_i \cdot \text{MRC}_i = \frac{w_i(\Sigma w)_i}{\sigma_p}
$$

And the key identity — these sum to total risk (Euler's theorem, since $\sigma_p$ is homogeneous of degree 1 in $w$):

In [2]:
def risk_contributions(w, sigma):
    """
    Diagnostic: the per-asset risk contributions RC_i = w_i (Sigma w)_i / sigma_p.
    They sum to sigma_p (Euler's theorem on the degree-1-homogeneous sigma_p),
    and for a correct equal-risk solution every RC_i should be ~equal.

    Returns (sigma_p, rc) where rc is the (n,) vector of contributions and
    rc / sigma_p is each asset's fractional share of total risk.
    """
    w = np.asarray(w, float).ravel()
    sigma = np.asarray(sigma, float)
    sigma_p = float(np.sqrt(w @ sigma @ w))
    rc = w * (sigma @ w) / sigma_p
    return sigma_p, rc

Equal risk contribution means every asset contributes the same share:

$$
\text{RC}_i = \text{RC}_j \quad \text{for all } i, j \implies w_i(\Sigma w)_i = w_j(\Sigma w)_j
$$

Equivalently, since the contributions must sum to $\sigma_p$ and there are $n$ of them, each asset should contribute exactly $1/n$ of total risk:

$$
w_i(\Sigma w)_i = \frac{\sigma_p^2}{n} \quad \text{for all } i
$$

This is a *system of nonlinear equations* in $w$ (the $w_i$ multiplies $(\Sigma w)_i$ which itself depends on all weights), so unlike Markowitz there's generally **no closed form** — you solve it numerically, typically as a convex optimization. That's a meaningful contrast with the closed forms, and worth knowing why: the equal-RC condition is quadratic-in-$w$ on both sides, not linear.

**The one special case that *does* have a closed form** — worth knowing because it's the sanity check. If all assets have **equal pairwise correlation** $\rho$, the risk parity weights are simply inversely proportional to volatility:

$$
w_i \propto \frac{1}{\sigma_i}
$$

This is the "naive risk parity" or "inverse-vol" portfolio. It's exact only under equal correlations, but it's widely used as an approximation and as the starting point for the numerical solver. It also captures the core intuition cleanly: **down-weight the volatile assets, up-weight the calm ones, so each contributes equally.**

**The convex formulation** (how the solver actually does it). Rather than solving the quadratic system directly, the standard approach minimizes a convex objective whose unique minimum is the risk parity portfolio:

$$
\min_w \ \frac{1}{2}w^T\Sigma w - \frac{\sigma_p^2}{n}\sum_{i=1}^n \ln w_i
$$

The clever part is the − $\sum \ln w_i$ **log-barrier** term. Its gradient contributes $-1/w_i$, and setting the full gradient to zero gives exactly $w_i(\Sigma w)_i = \text{const}$ — the equal-risk condition. The log barrier also forces $w_i > 0$ (long-only, since $⁡\ln$ of a non-positive number is undefined), which is intrinsic to standard risk parity. This objective is convex, so it has a unique solution reachable by Newton's method.

In [3]:
def risk_parity_fixed_point(sigma, b=None, tol=1e-10, max_iter=10000):
    """
    Risk parity via cyclical fixed-point iteration (Spinu / Griveau-Billion).
    Directly enforces w_i (Sigma w)_i = b_i by the update

        w_i  <-  b_i / (Sigma w)_i      (then renormalize)

    iterated to convergence. This is the lightweight scheme most production
    code actually uses.
    """
    sigma = np.asarray(sigma, float)
    n = sigma.shape[0]
    if b is None:
        b = np.ones(n) / n
    else:
        b = np.asarray(b, float).ravel()
        if np.any(b <= 0):
            raise ValueError("risk budgets must be strictly positive")
        b = b / b.sum()

    # Start from inverse-vol (the exact solution under equal correlation —
    # a good warm start that already captures the down-weight-the-volatile logic).
    vol = np.sqrt(np.diag(sigma))
    w = (1.0 / vol)
    w = w / w.sum()

    for i in range(max_iter):
        print(i)
        Sw = sigma @ w
        if np.any(Sw <= 0):
            # Individual entries of Σw can be negative if there are large enough negative off-diagonal entries in Σ.
            raise ValueError("Sigma w has non-positive entries; check Sigma is SPD")
        w_new = b / Sw                                 # the fixed-point step
        w_new = w_new / w_new.sum()                    # renormalize to sum 1
        if np.max(np.abs(w_new - w)) < tol:
            w = w_new
            break
        w = w_new
    else:
        raise ValueError(f"did not converge in {max_iter} iterations")

    return w

Fixed-point iteration (the $w_i ← b_i/(\Sigma w)_i$ scheme) is fastest and simplest for risk parity specifically, because it exploits the exact structure of the risk-contribution condition.
CVXPY + CLARABEL is best when you want readability, provable convexity, and the ability to add constraints (budgets, bounds) declaratively.

In [4]:
def risk_parity_cvxpy(sigma, b=None):
    """
    Risk parity (equal or budgeted risk contribution) via the convex
    log-barrier formulation:

        min  (1/2) w^T Sigma w  -  sum_i b_i * ln(w_i)

    The stationarity condition of this objective is w_i (Sigma w)_i = b_i * const,
    i.e. each asset's risk contribution is proportional to its budget b_i.
    Long-only by construction (ln requires w > 0). Weights are rescaled to sum to 1.

    Parameters
    ----------
    sigma : (n, n) covariance matrix, SPD. CLEAN IT FIRST (shrinkage / RMT) —
            risk parity has no mu, so a bad Sigma is the whole error budget.
    b     : (n,) risk budgets (positive, summing to 1). None => equal risk (b_i = 1/n).

    Returns
    -------
    w : (n,) risk parity weights, summing to 1.
    """
    sigma = np.asarray(sigma, float)
    n = sigma.shape[0]
    if b is None:
        b = np.ones(n) / n
    else:
        b = np.asarray(b, float).ravel()
        if np.any(b <= 0):
            raise ValueError("risk budgets must be strictly positive")
        b = b / b.sum()

    w = cp.Variable(n, pos=True)                      # pos=True enforces w > 0
    objective = cp.Minimize(
        0.5 * cp.quad_form(w, cp.psd_wrap(sigma)) - b @ cp.log(w)
    )
    prob = cp.Problem(objective)
    prob.solve(solver=cp.CLARABEL)                         # CLARABEL handle the log; OSQP cannot

    if prob.status not in ("optimal", "optimal_inaccurate"):
        raise ValueError(f"solve failed: {prob.status}")

    w_sol = np.asarray(w.value).ravel()
    return w_sol / w_sol.sum()                         # rescale to budget = 1

L-BFGS-B with analytic gradient is perfectly serviceable, and a reasonable choice if you're already in a scipy-only environment and don't want the CVXPY dependency. Not wrong, just not exploiting anything special.

In [5]:
def risk_parity(cov: np.ndarray) -> np.ndarray:
    """
    Risk parity portfolio via the convex log-barrier formulation:

        min  (1/2) wᵀΣw  -  (1/n) Σᵢ ln(wᵢ)

    The log-barrier forces wᵢ > 0 and its gradient contributes -1/(n·wᵢ),
    so the first-order condition gives wᵢ(Σw)ᵢ = const for all i — the
    equal risk contribution condition. Any positive constant gives the same
    optimal direction; we use 1/n and normalize at the end.

    Parameters
    ----------
    cov : (n, n) annualized covariance matrix

    Returns
    -------
    w : (n,) risk parity weights, long-only, sum to 1
    """
    n = cov.shape[0]

    def objective(w):
        return 0.5 * w @ cov @ w - np.sum(np.log(w)) / n

    def gradient(w):
        return cov @ w - 1.0 / (n * w)

    result = minimize(
        objective,
        x0=np.ones(n) / n,
        jac=gradient,
        method="L-BFGS-B",
        bounds=[(1e-8, None)] * n,
        options={"ftol": 1e-12, "gtol": 1e-8},
    )

    if not result.success:
        raise ValueError(f"solve failed: {result.message}")

    w = result.x / result.x.sum()
    return w

In [6]:
BUDGET = 100_000

tickers = ["SPY", "EFA", "EEM", "TLT", "HYG", "USO", "GLD", "IBIT"]
returns = uyf.get_returns(tickers, start="2024-01-01")
cov = returns.cov().values * 252
w = risk_parity_cvxpy(cov)

sigma_p, rc  = risk_contributions(w, cov)
prices       = uyf.get_latest_prices(tickers)
price_arr    = np.array([prices[t] for t in tickers])
dollar_alloc = w * BUDGET
shares       = np.floor(dollar_alloc / price_arr).astype(int)   # whole shares only
invested     = shares * price_arr

df = pd.DataFrame({
    "Weight":        w,
    "Risk Contrib":  rc / sigma_p,
    "Price":         price_arr,
    "Shares":        shares,
    "Invested ($)":  invested,
}, index=tickers)

fmt = {
    "Weight":       "{:.2%}".format,
    "Risk Contrib": "{:.2%}".format,
    "Price":        "${:,.2f}".format,
    "Shares":       "{:,d}".format,
    "Invested ($)": "${:,.0f}".format,
}
print(df.to_string(formatters=fmt))
print(f"\nPortfolio σ (annualised): {sigma_p:.2%}")
print(f"Budget: ${BUDGET:,}  |  Invested: ${invested.sum():,.0f}  |  Cash: ${BUDGET - invested.sum():,.0f}")

     Weight Risk Contrib   Price Shares Invested ($)
SPY   9.43%       12.50% $741.75     12       $8,901
EFA   9.25%       12.50% $105.02     88       $9,242
EEM   7.79%       12.50%  $67.88    114       $7,738
TLT  21.40%       12.50%  $85.77    249      $21,357
HYG  28.66%       12.50%  $79.94    358      $28,619
USO   9.85%       12.50% $125.43     78       $9,784
GLD   9.43%       12.50% $386.54     24       $9,277
IBIT  4.19%       12.50%  $36.04    116       $4,181

Portfolio σ (annualised): 8.72%
Budget: $100,000  |  Invested: $99,097  |  Cash: $903
